# StealthDiffusion — Milestone 2: making the attack transfer better and survive post-processing

**FIT5230 — Theme 1 (Adversarial ML on Generative AI), Dark side (improve an existing attack)**
**Team:** `Dark.ModelBreakers` — `<Member 1>`, `<Member 2>`

**Reference paper.** Z. Zhou, K. Sun, Z. Chen, H. Kuang, X. Sun, R. Ji.
*StealthDiffusion: Towards Evading Diffusion Forensic Detection through Diffusion Model.* ACM MM 2024. arXiv:2408.05669.
Official code: https://github.com/wyczzy/StealthDiffusion (we pin commit `a156fb0`).
Our Milestone 1 notebook (baseline reproduction): `StealthDiffusion_Baseline_Submission_1.ipynb`.

---
### What this notebook does
Milestone 1 reproduced the official attack (PGD -> latent adversarial optimisation (LAO) -> Control-VAE decode).
Milestone 2 changes the attack itself. All changes are switches of one script (`m2_stealth.py`), so every variant is run with the same data, weights, seeds and iteration count:

| Variant | Surrogate detectors in the loss | Where the LAO loss is computed | Objective | Extra terms |
|---|---|---|---|---|
| **B00** (official) | E only | ordinary VAE decode (official) | cross-entropy (official) | none |
| **B10** | E only | **final Control-VAE path** (new) | cross-entropy | none |
| **ENS** | **E + R + D ensemble** (new) | ordinary VAE decode | cross-entropy | none |
| **FULL** | E + R + D ensemble | final Control-VAE path | **margin loss** (new) | **LPIPS constraint + input diversity** (new) |

Detector **S (Swin)** is never used in any loss, so it is a true transfer target for all variants.

Sections: 3 setup (official code + weights) · 4 data · 5 attack runs · 6 evaluation (transfer, post-processing, image quality) · 7 challenge package for other teams · 8 engagement · 9 limitations.

> Runtime: Colab **A100** (40 GB). Free T4 is not enough for the diffusion back-propagation (see M1).
> Set the paths in Section 3.4 / 4.1, then *Run all*. With `N_ATTACK = 20` the four variants take roughly 40-80 minutes.

## 1. Problem statement
AI-Generated Content Stealth (AIGC-S): take an AI-generated image and change it so that a forensic detector (real vs. AI) calls it *real*, while the image still looks the same.
StealthDiffusion does this with Stable Diffusion: a small PGD step in pixel space, then optimisation of the SD latent against a surrogate detector, then decoding with a Control-VAE that removes the decoder's spectral fingerprint.

The paper leaves three things open, and these are what we attack in M2:

1. **Transfer is unstable.** With one surrogate detector, the paper's own Table 2 shows transfer ASR to UniDetection between 18.6% and 61.1% depending on the surrogate. The attack is optimised against a single model.
2. **The optimised image is not the saved image.** In the official code the LAO loss is computed on an ordinary-VAE decode of the latent (`diff_latent_attack.py`, line 435), but the image that is saved comes from the Control-VAE decoder (line 518-520). The attack can succeed on the image it optimises and still fail on the image it outputs.
3. **No post-processing evaluation.** All results in the paper are on lossless 224x224 PNG. Real images are JPEG-compressed and resized before a detector ever sees them. The paper reports PSNR/SSIM only, and LPIPS is computed in the code but never used.

## 2. What we change in the attack
Everything is inside our `m2_stealth.py`, which imports the official modules and re-implements only the LAO loop.

| Switch | Change | Why |
|---|---|---|
| `--surrogates E,R,D` | The PGD stage and the LAO loss use the **average** of several official detectors (EfficientNet-B0, ResNet-50, DeiT-B) instead of one. | Ensemble attacks transfer better in pixel space; here we move the idea into the latent space. |
| `--decode_path final` | The LAO loss is computed on the **Control-VAE decoding path** that produces the saved image (same weights, same 0.1825 scale, same PGD-derived control features). | The optimised image and the output image become the same image. |
| `--loss margin` | Objective `relu(kappa - (logit_real - logit_AI))` instead of `-CE`. | Pushes the sample *past* the decision boundary by a margin, so that a small threshold change or post-processing does not undo it. |
| `--lpips_weight 1.0` | Adds `LPIPS(output, input)` to the objective. | Keeps the perceptual change small while the attack becomes stronger. |
| `--di_prob 0.5` | Input diversity: with probability 0.5 the decoded image is randomly resized (90-100%) and padded before the surrogates see it. | Standard transfer trick (DI); also makes the perturbation less tied to one exact pixel grid. |

Not changed: DDIM inversion, 2 diffusion steps (`start_step 18` of 20), AdamW lr 1e-2, 10 LAO iterations (official code default), PGD eps 4/255 with 10 steps, Control-VAE weights, 224x224 resolution, VAE sampling seed 8888.

## 3. Setup: official code, environment, weights
Same procedure as Milestone 1: clone the official repository at a pinned commit, build a separate Python 3.10 environment (the official 2023 dependencies do not install on the current Colab Python), link the six author checkpoints from Google Drive, patch the hard-coded proxy / multi-GPU ids.

In [ ]:
#@title 3.1  Clone the official repository (pinned commit)
from pathlib import Path
import subprocess, sys, os

REPO = Path('/content/StealthDiffusion')
COMMIT = 'a156fb0792732e1aabc5b94a0cbecf6829e79d73'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/wyczzy/StealthDiffusion.git', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == COMMIT, f'unexpected commit {actual}; use a fresh runtime'
os.chdir(REPO)
print('repository at', REPO, 'commit', actual[:7])

In [ ]:
#@title 3.2  Separate Python 3.10 environment with the official dependency versions (~5-8 min)
import shutil
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
UV = shutil.which('uv')
PYTHON = '/content/sdenv/bin/python'
if not Path(PYTHON).exists():
    subprocess.run([UV, 'venv', '--python', '3.10', '/content/sdenv'], check=True)
subprocess.run([UV, 'pip', 'install', '--python', PYTHON, 'torch==2.2.2', 'torchvision==0.17.2',
                '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
packages = ['setuptools==80.9.0', 'numpy==1.26.4', 'diffusers==0.12.1', 'huggingface_hub==0.23.4',
            'transformers==4.30.2', 'accelerate==0.20.3', 'timm==0.6.12', 'efficientnet-pytorch==0.7.1',
            'torchattacks==3.5.1', 'lossers==0.0.9', 'eagerpy==0.30.0', 'natsort==8.4.0', 'tensorboardX==2.6.2.2',
            'scipy==1.11.4', 'scikit-learn==1.3.2', 'opencv-python-headless==4.10.0.84', 'Pillow==10.4.0',
            'tqdm==4.66.1', 'safetensors==0.4.3']
subprocess.run([UV, 'pip', 'install', '--python', PYTHON, *packages], check=True)
check = '''
import sys, torch, diffusers, timm, torchattacks, transformers
from diffusers import StableDiffusionPipeline
from lossers.lpips import LPIPS
assert sys.version_info[:2] == (3, 10), sys.version
assert torch.cuda.is_available(), "no CUDA GPU visible - choose an A100 runtime"
print("env OK | python", sys.version.split()[0], "| torch", torch.__version__, "| diffusers", diffusers.__version__,
      "| timm", timm.__version__, "| GPU", torch.cuda.get_device_name(0))
'''
result = subprocess.run([PYTHON, '-u', '-c', check], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(result.stdout[-3000:])
assert result.returncode == 0, 'environment check failed - fix the last error above before continuing'

In [ ]:
#@title 3.3  Get the six author checkpoints (link download OR Drive mount)
# EASIEST (no Drive mount, no account juggling): in Drive, right-click the `checkpoints`
# folder -> Share -> Anyone with the link (Viewer) -> Copy link, and paste it into
# CKPT_GDRIVE_LINK below. Leave it empty to use the Drive-mount path (CKPT_SRC) instead.
CKPT_GDRIVE_LINK = ""  #@param {type:"string"}
CKPT_SRC = "/content/drive/MyDrive/FIT5230_Ass1/StealthDiffusion/checkpoints"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/FIT5230_M2_out"          #@param {type:"string"}
import shutil, glob
os.makedirs('checkpoints', exist_ok=True)   # under /content/StealthDiffusion
need = ['efficientnet-b0.pth', 'resnet50.pth', 'deit.pth', 'swin-t.pth', 'Controlvae.pt', 'dncnn_color_blind.pth']

def link_link_files(src_dir):
    for f in need:
        s, d = os.path.join(src_dir, f), os.path.join('checkpoints', f)
        if os.path.exists(s) and not os.path.exists(d):
            try:
                os.symlink(s, d)
            except FileExistsError:
                pass

if CKPT_GDRIVE_LINK.strip():
    # ---- mount-free path: download the shared folder with gdown ----
    import subprocess as _sp
    _sp.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'gdown'], check=False)
    import gdown
    dl = '/content/_ckpt_dl'
    shutil.rmtree(dl, ignore_errors=True)
    gdown.download_folder(CKPT_GDRIVE_LINK.strip(), output=dl, quiet=False, use_cookies=False)
    for f in need:
        hits = glob.glob(os.path.join(dl, '**', f), recursive=True)
        if hits and not os.path.exists(os.path.join('checkpoints', f)):
            shutil.copy(hits[0], os.path.join('checkpoints', f))
    # outputs cannot go to Drive if it is not mounted; keep them local
    if DRIVE_ROOT.startswith('/content/drive') and not os.path.ismount('/content/drive'):
        DRIVE_ROOT = '/content/M2_out'
        print('[3.3] Drive not mounted; outputs will go to local', DRIVE_ROOT,
              '(download them from the Files panel, or zip at the end).')
else:
    # ---- Drive-mount path ----
    from google.colab import drive
    # Mount FIRST and robustly. Never create anything under /content/drive before it is
    # mounted, or later mounts fail with "Mountpoint must not already contain files".
    if not os.path.ismount('/content/drive'):
        try:
            drive.mount('/content/drive', force_remount=True)
        except Exception as e:
            print('first mount attempt failed, clearing stale /content/drive and retrying:', e)
            if os.path.isdir('/content/drive') and not os.path.ismount('/content/drive'):
                shutil.rmtree('/content/drive', ignore_errors=True)
            drive.mount('/content/drive', force_remount=True)
    assert os.path.ismount('/content/drive'), (
        'Google Drive did not mount. Use the CKPT_GDRIVE_LINK option above, or '
        'Runtime -> Disconnect and delete runtime and re-run.')

    def find_ckpt_dir(root='/content/drive/MyDrive', need=need, max_depth=6):
        # Search the mounted My Drive for the folder that holds the checkpoints.
        if not os.path.isdir(root):
            return None
        need_set = set(need)
        for dp, dns, fns in os.walk(root):
            if dp[len(root):].strip('/').count('/') >= max_depth:
                dns[:] = []
            if len(need_set & set(fns)) >= 5:
                return dp
        return None

    if not all(os.path.exists(os.path.join(CKPT_SRC, f)) for f in need):
        print(f"[3.3] not all checkpoints at CKPT_SRC={CKPT_SRC}; searching this account's My Drive ...", flush=True)
        found = find_ckpt_dir()
        if found:
            print(f"[3.3] found checkpoints at: {found}")
            CKPT_SRC = found
        else:
            top = sorted(os.listdir('/content/drive/MyDrive'))[:40] if os.path.isdir('/content/drive/MyDrive') else []
            print("[3.3] could not find the checkpoints in this account's My Drive.")
            print("      top-level of the mounted My Drive:", top)
            print("      -> This Colab is mounting an account whose My Drive does NOT contain them.")
            print("         Easiest fix: use the CKPT_GDRIVE_LINK option at the top of this cell.")
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    link_link_files(CKPT_SRC)

os.makedirs(DRIVE_ROOT, exist_ok=True)
missing = [f for f in need if not os.path.exists(os.path.join('checkpoints', f))]
for f in need:
    print(f"{f:<26}{'READY' if f not in missing else 'MISSING':>9}")
assert not missing, ('checkpoints missing; paste the checkpoints folder share link into '
                     'CKPT_GDRIVE_LINK at the top of this cell, or fix CKPT_SRC.')

In [ ]:
#@title 3.4  Single-GPU patch (proxy lines and cuda:1/2/3 -> cuda:0), same as M1
import re, pathlib
proxy_re = re.compile(r'^(\s*)(os\.environ\[["\']https?_proxy["\']\]\s*=)')
cvd_re   = re.compile(r'(CUDA_VISIBLE_DEVICES["\']\]\s*=\s*["\'])([1-9])(["\'])')
dev_re   = re.compile(r'cuda:([1-9])')
n = 0
for f in pathlib.Path('.').rglob('*.py'):
    if f.name.startswith('m2_'):
        continue
    out, changed = [], False
    for ln in f.read_text().splitlines(keepends=True):
        o = ln
        if proxy_re.match(ln) and not ln.lstrip().startswith('#'): ln = re.sub(proxy_re, r'\1# \2', ln)
        if cvd_re.search(ln): ln = cvd_re.sub(r'\g<1>0\g<3>', ln)
        if dev_re.search(ln): ln = dev_re.sub('cuda:0', ln)
        out.append(ln); changed = changed or (ln != o)
    if changed: f.write_text(''.join(out)); n += 1
print('patched files:', n)

### 3.5 Our M2 code
The next cells write our scripts into the repository directory. They run inside the Python 3.10 environment (`/content/sdenv/bin/python`) because they import the official modules.

- `m2_data.py` — real photos (Imagenette) + Stable Diffusion images, detector sanity check
- `m2_stealth.py` — the attack with the M2 switches (defaults = official pipeline)
- `m2_evaluate.py` — evaluation of the saved PNGs: transfer, post-processing, quality, AUC
- `m2_challenge.py` — challenge package for other teams + scorer

In [ ]:
%%writefile /content/StealthDiffusion/m2_device.py
"""Device compatibility shim so the CUDA-only StealthDiffusion code runs on Apple Silicon (MPS) or CPU.

The official code and our scripts call `.cuda()` in many places. Calling `enable(device)`
once at start-up redirects every `.cuda()` (on tensors and modules, including the upstream
code we import) to the chosen device, and turns the CUDA-only bookkeeping calls
(`torch.cuda.synchronize` / `empty_cache` / `manual_seed*`) into no-ops when there is no CUDA.

Usage (already wired into m2_stealth.py / m2_data.py / m2_evaluate.py):
    import m2_device
    DEV = m2_device.enable("auto")   # "auto" -> cuda if available, else mps, else cpu
"""
import os

import torch

DEVICE = "cuda"


def resolve(name="auto"):
    if name and name != "auto":
        return name
    env = os.environ.get("M2_DEVICE", "").strip()
    if env:
        return env
    if torch.cuda.is_available():
        return "cuda"
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return "mps"
    return "cpu"


def local_model(model_id, ignore_fp16=True):
    """Pre-fetch a Hugging Face model, downloading only ONE weight variant, and return a local path.

    diffusers 0.12.1 downloads every variant a repo contains (fp32 `.bin` AND `.fp16.bin`
    AND `non_ema`), which is roughly twice the bytes actually needed. We fetch with ignore
    patterns and hand `from_pretrained` a local folder, so it does not download again.
    Falls back to the plain model id if anything goes wrong.
    """
    if os.path.isdir(model_id):
        return model_id
    try:
        import glob
        from huggingface_hub import snapshot_download
    except ImportError:
        return model_id
    base = ["*.msgpack", "*.onnx*", "*.ckpt", "*non_ema*", "*.flax*", "*.h5"]
    ignore = base + ["*.fp16.*"] if ignore_fp16 else base
    try:
        path = snapshot_download(model_id, ignore_patterns=ignore)
    except Exception as e:  # noqa: BLE001
        print(f"[m2_device] prefetch of {model_id} failed ({e}); letting from_pretrained handle it", flush=True)
        return model_id
    # Safety net: if the repo only ships fp16 weights, re-fetch without that filter.
    unet = os.path.join(path, "unet")
    if ignore_fp16 and os.path.isdir(unet) and not (
            glob.glob(os.path.join(unet, "*.bin")) or glob.glob(os.path.join(unet, "*.safetensors"))):
        print("[m2_device] no full-precision weights in repo; re-fetching including fp16", flush=True)
        try:
            path = snapshot_download(model_id, ignore_patterns=base)
        except Exception:  # noqa: BLE001
            return model_id
    print(f"[m2_device] model ready at {path}", flush=True)
    return path


def enable(name="auto"):
    """Resolve and install the device shim. Returns the device string."""
    global DEVICE
    DEVICE = resolve(name)
    dev = DEVICE
    if dev == "mps":
        # let unimplemented MPS ops fall back to CPU instead of erroring
        os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
    if dev != "cuda":
        torch.Tensor.cuda = lambda self, *a, **k: self.to(dev)      # type: ignore[assignment]
        torch.nn.Module.cuda = lambda self, *a, **k: self.to(dev)   # type: ignore[assignment]
        if not torch.cuda.is_available():
            torch.cuda.synchronize = lambda *a, **k: None           # type: ignore[assignment]
            torch.cuda.empty_cache = lambda *a, **k: None           # type: ignore[assignment]
            torch.cuda.manual_seed = lambda *a, **k: None           # type: ignore[assignment]
            torch.cuda.manual_seed_all = lambda *a, **k: None       # type: ignore[assignment]
    print(f"[m2_device] using device: {dev}", flush=True)
    return dev

In [ ]:
%%writefile /content/StealthDiffusion/m2_data.py
"""
FIT5230 · Dark.ModelBreakers · Milestone 2 — data preparation.

Run from inside the official StealthDiffusion checkout with its Python 3.10 environment.

  python m2_data.py real     --out data --n_dev 10 --n_test 100
      Downloads Imagenette (a public 10-class subset of ImageNet, real photographs),
      takes validation images round-robin over the 10 classes, centre-crops and resizes
      them to 224x224 PNG:  data/real/dev/*.png  and  data/real/test/*.png

  python m2_data.py generate --out data --n_dev 10 --n_test 100 --model stable-diffusion-v1-5/stable-diffusion-v1-5
      Generates AI images with Stable Diffusion for the same 10 classes
      ("a photo of a <class>"), 512x512, then saves 224x224 PNG copies:
      data/gen/dev/*.png, data/gen/test/*.png  (512px originals in data/gen512/)

  python m2_data.py check    --out data
      Runs the four official detectors (E,R,D,S) on the dev sets and prints the clean
      accuracy and the label polarity (official convention: class 0 = AI, class 1 = real).

Every image is listed in a manifest CSV with its origin (file path or prompt+seed+model).
"""
import argparse
import csv
import glob
import os
import shutil
import sys
import tarfile
import urllib.request

import numpy as np
from PIL import Image
from natsort import natsorted, ns

IMAGENETTE_URLS = [
    "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz",
    "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz",
]
IMAGENETTE_CLASSES = [
    ("n01440764", "tench"), ("n02102040", "English springer spaniel"), ("n02979186", "cassette player"),
    ("n03000684", "chain saw"), ("n03028079", "church"), ("n03394916", "French horn"),
    ("n03417042", "garbage truck"), ("n03425413", "gas pump"), ("n03445777", "golf ball"),
    ("n03888605", "parachute"),
]
RES = 224


def center_square_resize(pil, res=RES):
    pil = pil.convert("RGB")
    w, h = pil.size
    s = min(w, h)
    left, top = (w - s) // 2, (h - s) // 2
    return pil.crop((left, top, left + s, top + s)).resize((res, res), resample=Image.LANCZOS)


def write_manifest(path, rows, fields):
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)


def _imagenette_root(out):
    for d in glob.glob(os.path.join(out, "imagenette2*")):
        if os.path.isdir(os.path.join(d, "val")):
            return d
    return None


def download_imagenette(out):
    """Download + extract Imagenette robustly: tries several URLs, atomic download, re-download on a bad archive."""
    root = _imagenette_root(out)
    if root:
        return root
    os.makedirs(out, exist_ok=True)
    last_err = None
    for url in IMAGENETTE_URLS:
        tgz = os.path.join(out, os.path.basename(url))
        for attempt in range(2):
            try:
                if not os.path.exists(tgz) or os.path.getsize(tgz) < 50 * 1024 * 1024:
                    print(f"[data] downloading {url} (attempt {attempt + 1})", flush=True)
                    part = tgz + ".part"
                    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
                    with urllib.request.urlopen(req, timeout=180) as r, open(part, "wb") as f:
                        shutil.copyfileobj(r, f)
                    os.replace(part, tgz)
                print("[data] extracting", flush=True)
                with tarfile.open(tgz) as tf:
                    tf.extractall(out)
                root = _imagenette_root(out)
                if root:
                    return root
            except Exception as e:  # noqa: BLE001 - report and try the next URL/attempt
                last_err = e
                print(f"[data] {url} failed: {e}", flush=True)
                if os.path.exists(tgz):
                    os.remove(tgz)
    raise RuntimeError(
        f"Imagenette download failed ({last_err}). Pass --real_src <folder of real images> to skip it "
        "(e.g. REAL_SRC=/path bash run_local_mac.sh data).")


def cmd_real(args):
    names = dict(IMAGENETTE_CLASSES)
    # Escape hatch: use an existing folder of real photos instead of downloading Imagenette.
    if args.real_src:
        src = args.real_src
        if not os.path.isdir(src):
            raise FileNotFoundError(f"--real_src folder not found: {src}")
        files = natsorted([os.path.join(dp, f) for dp, _, fs in os.walk(src)
                           for f in fs if f.lower().endswith((".jpeg", ".jpg", ".png"))], alg=ns.PATH)
        if len(files) < args.n_dev + args.n_test:
            raise ValueError(f"--real_src has only {len(files)} images (< {args.n_dev + args.n_test}); "
                             f"add more or lower N_DEV/N_TEST")
        rng = np.random.default_rng(args.seed)
        rng.shuffle(files)
        order = [(None, f) for f in files[: args.n_dev + args.n_test]]
        print(f"[data] using {len(order)} real images from {src}", flush=True)
    else:
        root = download_imagenette(args.out)
        val_root = os.path.join(root, "val")
        # Use whatever class folders actually exist (robust to a missing class / varied layout).
        class_dirs = sorted(d for d in os.listdir(val_root) if os.path.isdir(os.path.join(val_root, d)))
        if not class_dirs:
            raise RuntimeError(f"no class folders under {val_root}")
        per_class = {}
        for ci, wnid in enumerate(class_dirs):
            files = natsorted([f for f in os.listdir(os.path.join(val_root, wnid)) if f.lower().endswith((".jpeg", ".jpg", ".png"))], alg=ns.PATH)
            rng = np.random.default_rng(args.seed + ci)
            rng.shuffle(files)
            per_class[wnid] = [os.path.join(val_root, wnid, f) for f in files]
        # round-robin over classes so every split is class balanced
        order = []
        i = 0
        need_total = args.n_dev + args.n_test
        while len(order) < need_total and any(i < len(per_class[w]) for w in class_dirs):
            for wnid in class_dirs:
                if i < len(per_class[wnid]):
                    order.append((wnid, per_class[wnid][i]))
            i += 1
        print(f"[data] {len(class_dirs)} real classes found; collected {len(order)} candidate images", flush=True)

    # Save images; skip any that fail to open (e.g. a truncated download) and pull the next one.
    def save_split(out_dir, items):
        os.makedirs(out_dir, exist_ok=True)
        rows = []
        for wnid, path in items:
            if len(rows) >= out_dir_target[out_dir]:
                break
            try:
                img = center_square_resize(Image.open(path))
            except Exception as e:  # noqa: BLE001
                print(f"[data] skip unreadable {path}: {e}", flush=True)
                continue
            image_id = str(len(rows)).rjust(4, "0")
            img.save(os.path.join(out_dir, image_id + ".png"))
            try:
                source = os.path.relpath(path, args.out)
            except ValueError:
                source = path
            rows.append({"image_id": image_id, "class": names.get(wnid, "unknown"),
                         "wnid": wnid or "", "source": source})
        write_manifest(os.path.join(out_dir, "manifest.csv"), rows, ["image_id", "class", "wnid", "source"])
        print(f"[data] {out_dir}: {len(rows)} images", flush=True)
        return len(rows)

    dev_dir = os.path.join(args.out, "real", "dev")
    test_dir = os.path.join(args.out, "real", "test")
    out_dir_target = {dev_dir: args.n_dev, test_dir: args.n_test}
    n_dev_saved = save_split(dev_dir, order[: args.n_dev])
    n_test_saved = save_split(test_dir, order[args.n_dev:])
    if n_dev_saved < args.n_dev or n_test_saved < args.n_test:
        print(f"[data] WARNING: wanted dev {args.n_dev}/test {args.n_test}, got {n_dev_saved}/{n_test_saved}. "
              "If the archive looks truncated, delete data/imagenette2-320* and re-run, or use --real_src.", flush=True)


def cmd_generate(args):
    import torch
    import m2_device
    from diffusers import StableDiffusionPipeline

    dev = m2_device.enable(args.device)
    # fp16 only helps on CUDA; MPS/CPU are more reliable in fp32
    kwargs = {"torch_dtype": torch.float16 if dev == "cuda" else torch.float32, "safety_checker": None}
    model_path = m2_device.local_model(args.model, ignore_fp16=(dev != "cuda"))
    try:
        pipe = StableDiffusionPipeline.from_pretrained(model_path, requires_safety_checker=False, **kwargs)
    except TypeError:
        pipe = StableDiffusionPipeline.from_pretrained(model_path, **kwargs)
    pipe = pipe.to(dev)
    pipe.set_progress_bar_config(disable=True)
    total = args.n_dev + args.n_test
    names = [n for _, n in IMAGENETTE_CLASSES]
    prompts = [f"a photo of a {names[k % len(names)]}" for k in range(total)]
    seeds = [args.seed * 1000 + k for k in range(total)]
    out512 = os.path.join(args.out, "gen512")
    os.makedirs(out512, exist_ok=True)
    gen_device = dev if dev in ("cuda", "mps") else "cpu"
    rows = []
    for start in range(0, total, args.batch_size):
        idx = list(range(start, min(start + args.batch_size, total)))
        gens = [torch.Generator(device=gen_device).manual_seed(seeds[k]) for k in idx]
        with torch.no_grad():
            images = pipe([prompts[k] for k in idx], num_inference_steps=args.steps, guidance_scale=args.guidance,
                          height=512, width=512, generator=gens).images
        for k, im in zip(idx, images):
            image_id = str(k).rjust(4, "0")
            im.save(os.path.join(out512, image_id + ".png"))
            rows.append({"image_id": image_id, "prompt": prompts[k], "seed": seeds[k], "model": args.model,
                         "steps": args.steps, "guidance": args.guidance})
        print(f"[data] generated {min(start + args.batch_size, total)}/{total}", flush=True)
    for split, lo, hi in [("dev", 0, args.n_dev), ("test", args.n_dev, total)]:
        out_dir = os.path.join(args.out, "gen", split)
        os.makedirs(out_dir, exist_ok=True)
        split_rows = []
        for k in range(lo, hi):
            src = str(k).rjust(4, "0")
            image_id = str(k - lo).rjust(4, "0")
            Image.open(os.path.join(out512, src + ".png")).resize((RES, RES), resample=Image.LANCZOS).save(
                os.path.join(out_dir, image_id + ".png"))
            r = dict(rows[k]); r["image_id"] = image_id; r["source512"] = f"gen512/{src}.png"
            split_rows.append(r)
        write_manifest(os.path.join(out_dir, "manifest.csv"), split_rows,
                       ["image_id", "prompt", "seed", "model", "steps", "guidance", "source512"])
        print(f"[data] gen/{split}: {len(split_rows)} images -> {out_dir}", flush=True)


def cmd_check(args):
    import torch
    from torchvision import transforms
    import other_attacks
    import m2_device
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    m2_device.enable(args.device)
    normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    def load_dir(d):
        files = natsorted([f for f in os.listdir(d) if f.endswith(".png")], alg=ns.PATH)
        x = np.stack([np.float32(Image.open(os.path.join(d, f)).convert("RGB")) / 255.0 for f in files])
        return torch.from_numpy(x).permute(0, 3, 1, 2)

    real = load_dir(os.path.join(args.out, "real", "dev"))
    gen = load_dir(os.path.join(args.out, "gen", "dev"))
    print(f"[check] dev real {tuple(real.shape)} | dev generated {tuple(gen.shape)}")
    print(f"{'detector':<10}{'acc(real->1)':>14}{'acc(gen->0)':>13}{'margin real':>13}{'margin gen':>12}")
    rows = []
    for name in args.detectors.split(","):
        det = other_attacks.model_selection(name).eval()
        det.requires_grad_(False)
        out = {}
        for tag, x in [("real", real), ("gen", gen)]:
            preds, margins = [], []
            with torch.no_grad():
                for i in range(0, len(x), 16):
                    logits = det(normalize(x[i:i + 16].cuda()))
                    preds += torch.argmax(logits, 1).tolist()
                    margins += (logits[:, 1] - logits[:, 0]).tolist()
            out[tag] = (np.array(preds), np.array(margins))
        acc_real = float((out["real"][0] == 1).mean())
        acc_gen = float((out["gen"][0] == 0).mean())
        print(f"{name:<10}{acc_real:>14.2f}{acc_gen:>13.2f}{out['real'][1].mean():>13.2f}{out['gen'][1].mean():>12.2f}")
        if acc_real < 0.5 and acc_gen < 0.5:
            print(f"   WARNING: {name} looks label-inverted on this data; check the checkpoint/polarity")
        rows.append({"detector": name, "acc_real": acc_real, "acc_gen": acc_gen,
                     "margin_real": float(out["real"][1].mean()), "margin_gen": float(out["gen"][1].mean())})
        del det
        torch.cuda.empty_cache()
    write_manifest(os.path.join(args.out, "dev_check.csv"), rows,
                   ["detector", "acc_real", "acc_gen", "margin_real", "margin_gen"])


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("mode", choices=["real", "generate", "check"])
    p.add_argument("--out", default="data")
    p.add_argument("--n_dev", type=int, default=10)
    p.add_argument("--n_test", type=int, default=100)
    p.add_argument("--seed", type=int, default=5230)
    p.add_argument("--real_src", default="",
                   help="folder of real photos to use instead of downloading Imagenette (searched recursively)")
    p.add_argument("--device", default="auto", choices=["auto", "cuda", "mps", "cpu"],
                   help="auto = cuda if available, else mps (Apple Silicon), else cpu")
    p.add_argument("--model", default="stable-diffusion-v1-5/stable-diffusion-v1-5",
                   help="Hugging Face id of the generator (alternative: Manojb/stable-diffusion-2-1-base)")
    p.add_argument("--steps", type=int, default=30)
    p.add_argument("--guidance", type=float, default=7.5)
    p.add_argument("--batch_size", type=int, default=4)
    p.add_argument("--detectors", default="E,R,D,S")
    args = p.parse_args(argv)
    {"real": cmd_real, "generate": cmd_generate, "check": cmd_check}[args.mode](args)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/StealthDiffusion/m2_stealth.py
"""
FIT5230 · Dark.ModelBreakers · Milestone 2
Modified StealthDiffusion attack (PGD -> Latent Adversarial Optimisation -> Control-VAE).

Run this file from inside the official StealthDiffusion checkout (pinned commit a156fb0)
with the same Python 3.10 environment the official code uses.  It imports the official
modules (diff_latent_attack, ControlVAE, other_attacks) and re-implements only the LAO
loop, so that every change is an explicit command-line switch:

  --surrogates E          detectors used in the PGD stage and in the LAO loss.
                          A comma list (e.g. E,R,D) turns on the M2 *surrogate ensemble*.
  --loss ce|margin        ce     = official objective  -CE(logits/10, label=AI) * 10
                          margin = M2 objective        relu(kappa - (logit_real - logit_ai))
  --decode_path plain|final
                          plain = official: the LAO loss is computed on the ordinary
                                  VAE decode, the saved image comes from Control-VAE.
                          final = M2: the LAO loss is computed on the *same* Control-VAE
                                  decoding path that produces the saved image.
  --lpips_weight 0        M2: weight of LPIPS(decoded image, input) inside the LAO loss.
                          (The official loop computes LPIPS but never uses it.)
  --di_prob 0             M2: input diversity - with this probability the decoded image is
                          randomly resized and padded before it is shown to the surrogates.

With all defaults the script reproduces the official pipeline (variant "B00").
Everything else (DDIM inversion, 2 diffusion steps, AdamW lr 1e-2, Control-VAE weights,
the two PGD stages, 224x224 resolution, seed 8888 for the VAE sampling) is kept as in
the official code so that variants are comparable.

Outputs per image (in --save_dir):
  NNNN_originImage.png   input resized to 224x224 (the reference for PSNR/SSIM/LPIPS)
  NNNN_plain_vae.png     ordinary-VAE decode of the final latent (diagnostic only)
  NNNN_adv_image.png     final Control-VAE output = the attack result
  records.jsonl          one line per image: predictions of all detectors, PSNR, SSIM, time
  run_config.json        the exact configuration of this run
"""
import argparse
import glob
import json
import os
import random
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
from natsort import natsorted, ns
import torchattacks

# The official other_attacks.py hides all GPUs but "1" at import time (the authors used a
# multi-GPU server).  Import it first, then restore the single visible Colab GPU.
import other_attacks  # noqa: E402
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import m2_device  # noqa: E402

from diffusers import StableDiffusionPipeline, DDIMScheduler  # noqa: E402
from lossers.lpips import LPIPS  # noqa: E402
from diff_latent_attack import (  # noqa: E402
    ddim_reverse_sample, diffusion_step, preprocess, calculate_psnr, calculate_ssim,
)
from ControlVAE import NewEncoder, NewDecoder  # noqa: E402

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
normalize = transforms.Normalize(mean=MEAN, std=STD)
denormalize = transforms.Normalize(mean=[-m / s for m, s in zip(MEAN, STD)], std=[1 / s for s in STD])

DETECTOR_NAMES = {"E": "efficientnet-b0", "R": "resnet50", "D": "deit-base", "S": "swin-base"}

# Control-VAE architecture, copied from the official diff_latent_attack.py
CVAE_PARAMS = {
    "act_fn": "silu",
    "block_out_channels": [128, 256, 512, 512],
    "down_block_types": ["DownEncoderBlock2D"] * 4,
    "in_channels": 3,
    "latent_channels": 4,
    "layers_per_block": 2,
    "norm_num_groups": 32,
    "out_channels": 3,
    "up_block_types": ["UpDecoderBlock2D"] * 4,
}
CVAE_ENCODER_PARAMS = {
    "in_channels": CVAE_PARAMS["in_channels"],
    "out_channels": CVAE_PARAMS["latent_channels"],
    "down_block_types": CVAE_PARAMS["down_block_types"],
    "block_out_channels": CVAE_PARAMS["block_out_channels"],
    "layers_per_block": CVAE_PARAMS["layers_per_block"],
    "act_fn": CVAE_PARAMS["act_fn"],
    "norm_num_groups": CVAE_PARAMS["norm_num_groups"],
    "double_z": True,
}
CVAE_DECODER_PARAMS = {
    "in_channels": CVAE_PARAMS["latent_channels"],
    "out_channels": CVAE_PARAMS["out_channels"],
    "up_block_types": CVAE_PARAMS["up_block_types"],
    "block_out_channels": CVAE_PARAMS["block_out_channels"],
    "layers_per_block": CVAE_PARAMS["layers_per_block"],
    "act_fn": CVAE_PARAMS["act_fn"],
    "norm_num_groups": CVAE_PARAMS["norm_num_groups"],
}
PLAIN_SCALE = 0.18215   # scale used by the official LAO loop (ordinary VAE)
FINAL_SCALE = 0.1825    # scale used by the official final Control-VAE decode (kept as is)


def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


class Ensemble(nn.Module):
    """Average of the official [0, logit_real] detector outputs -> used by the PGD stage."""

    def __init__(self, models):
        super().__init__()
        self.models = nn.ModuleList(models)

    def forward(self, x):
        return torch.stack([m(x) for m in self.models], dim=0).mean(dim=0)


def real_margin(logits):
    """Official detector wrapper returns [0, raw]; argmax==1 means 'real'. Margin = raw."""
    return logits[:, 1] - logits[:, 0]


def input_diversity(x, prob, low_ratio=0.9):
    """Random resize + zero pad back to the original size (differentiable), DI-style."""
    if prob <= 0 or torch.rand(1).item() > prob:
        return x
    h, w = x.shape[-2:]
    new_h = int(torch.randint(int(h * low_ratio), h + 1, (1,)).item())
    new_w = int(torch.randint(int(w * low_ratio), w + 1, (1,)).item())
    resized = F.interpolate(x, size=(new_h, new_w), mode="bilinear", align_corners=False)
    pad_h, pad_w = h - new_h, w - new_w
    top = int(torch.randint(0, pad_h + 1, (1,)).item())
    left = int(torch.randint(0, pad_w + 1, (1,)).item())
    return F.pad(resized, (left, pad_w - left, top, pad_h - top), value=0.0)


def attack_objective(logits_list, label, loss_type, kappa, attack_loss_weight):
    """LAO objective averaged over surrogate detectors (lower = better for the attacker)."""
    losses = []
    for logits in logits_list:
        if loss_type == "ce":
            # official: pred = classifier(x)/10 ; loss = -CE(pred, label) * attack_loss_weight
            losses.append(-F.cross_entropy(logits / 10, label) * attack_loss_weight)
        elif loss_type == "margin":
            losses.append(F.relu(kappa - real_margin(logits)).mean())
        else:
            raise ValueError(loss_type)
    return torch.stack(losses).mean()


def build_lpips():
    return LPIPS(net="vgg", cache_dir="./.cache").cuda()


def load_control_vae(model, encoder_weights):
    newencoder = NewEncoder(**CVAE_ENCODER_PARAMS)
    state = torch.load(encoder_weights, map_location="cpu")
    newencoder.load_state_dict(state["encoder"], strict=False)
    newencoder.requires_grad_(False)
    newencoder = newencoder.cuda().eval()
    decoder = NewDecoder(**CVAE_DECODER_PARAMS)
    decoder.load_state_dict(model.vae.decoder.state_dict(), strict=False)
    decoder.requires_grad_(False)
    decoder = decoder.cuda().eval()
    return newencoder, decoder


def to_uint8(img01):
    return np.uint8(img01.detach().clamp(0, 1).cpu().numpy()[0].transpose(1, 2, 0) * 255.0)


@torch.no_grad()
def predict(detectors, img01):
    x = normalize(img01)
    out = {}
    for name, det in detectors.items():
        logits = det(x)
        out[name] = {"pred": int(torch.argmax(logits, 1).item()),
                     "real_margin": float(real_margin(logits).item())}
    return out


def make_pgd(model_for_pgd, eps):
    attack = torchattacks.PGD(model_for_pgd, eps=eps, alpha=4 / 255, steps=10, random_start=False)
    attack.set_normalization_used(mean=MEAN, std=STD)
    return attack


@torch.enable_grad()
def attack_one(model, pil_image, args, surrogate_models, detectors, cvae, lpips_fn, out_prefix):
    """Attack one PIL image. Returns a record dict. Mirrors diff_latent_attack.diffattack."""
    res = args.res
    label = torch.tensor([0]).cuda()  # official label convention: 0 = AI-generated
    pgd_model = surrogate_models[0] if len(surrogate_models) == 1 else Ensemble(surrogate_models)

    # ---------- stage 0: PGD in pixel space (official) ----------
    origin_pil = pil_image.resize((res, res), resample=Image.LANCZOS)
    origin01 = torch.from_numpy(np.float32(origin_pil)[:, :, :3] / 255.0).permute(2, 0, 1)[None].cuda()
    test_image = normalize(origin01)
    pgd = make_pgd(pgd_model, args.pgd_eps)
    adv_norm = pgd(test_image, label)
    pgd_pil = Image.fromarray(to_uint8(denormalize(adv_norm)))

    # ---------- stage 1: DDIM inversion (official) ----------
    prompt = [""] * 2
    _, inversion_latents = ddim_reverse_sample(pgd_pil, prompt, model, args.diffusion_steps, 0, res=res)
    inversion_latents = inversion_latents[::-1]
    latent = inversion_latents[args.start_step - 1].detach().clone()

    uncond_input = model.tokenizer([""], padding="max_length", max_length=77, return_tensors="pt")
    uncond_embeddings = model.text_encoder(uncond_input.input_ids.to(model.device))[0]
    text_input = model.tokenizer(prompt, padding="max_length", max_length=model.tokenizer.model_max_length,
                                 truncation=True, return_tensors="pt")
    text_embeddings = model.text_encoder(text_input.input_ids.to(model.device))[0]
    timesteps = model.scheduler.timesteps[1 + args.start_step - 1:]
    context = [torch.cat([torch.cat([uncond_embeddings.detach()] * 2), text_embeddings]) for _ in timesteps]

    original_latent = latent.clone()
    latent.requires_grad_(True)
    optimizer = torch.optim.AdamW([latent], lr=args.lr)

    init_image = preprocess(pgd_pil, res)          # PGD image in [-1, 1]
    init_image_compare = preprocess(origin_pil, res)  # clean input in [-1, 1]

    # ---------- Control-VAE conditioning (official final path, computed once) ----------
    newencoder, decoder = cvae
    pgd2 = make_pgd(pgd_model, args.pgd_eps)          # second PGD stage of the official final path
    adv2 = pgd2(normalize(init_image * 0.5 + 0.5), label)
    init_image1 = (2.0 * denormalize(adv2) - 1).detach()
    with torch.no_grad():
        down_features = [f.detach() for f in newencoder(init_image1)[1]]

    def decode_plain(latents):
        return model.vae.decode(1 / PLAIN_SCALE * latents)["sample"][1:]

    def decode_final(latents):
        return decoder(model.vae.post_quant_conv(1 / FINAL_SCALE * latents), down_features)[1:]

    decode_for_loss = decode_final if args.decode_path == "final" else decode_plain

    # ---------- stage 2: latent adversarial optimisation ----------
    history = []
    for it in range(args.iterations):
        latents = torch.cat([original_latent, latent])
        for ind, t in enumerate(timesteps):
            latents = diffusion_step(model, latents, context[ind], t, args.guidance)
        dec = decode_for_loss(latents)                       # [-1, 1]
        out01 = (dec / 2 + 0.5).clamp(0, 1)
        x_in = normalize(input_diversity(out01, args.di_prob))
        logits_list = [m(x_in) for m in surrogate_models]
        loss_attack = attack_objective(logits_list, label, args.loss, args.kappa, args.attack_loss_weight)
        loss = loss_attack
        loss_lpips = torch.zeros(())
        if args.lpips_weight > 0:
            loss_lpips = lpips_fn(init_image_compare, dec).mean()
            loss = loss + args.lpips_weight * loss_lpips
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append({"iter": it, "loss": float(loss.item()), "attack": float(loss_attack.item()),
                        "lpips": float(loss_lpips.item()),
                        "margins": [float(real_margin(l).item()) for l in logits_list]})

    # ---------- final decode (official) ----------
    with torch.no_grad():
        latents = torch.cat([original_latent, latent.detach()])
        for ind, t in enumerate(timesteps):
            latents = diffusion_step(model, latents, context[ind], t, args.guidance)
        plain01 = (decode_plain(latents) / 2 + 0.5).clamp(0, 1)
        final01 = (decode_final(latents) / 2 + 0.5).clamp(0, 1)

    origin_u8, plain_u8, final_u8 = to_uint8(origin01), to_uint8(plain01), to_uint8(final01)
    Image.fromarray(origin_u8).save(out_prefix + "_originImage.png")
    Image.fromarray(plain_u8).save(out_prefix + "_plain_vae.png")
    Image.fromarray(final_u8).save(out_prefix + "_adv_image.png")

    record = {
        "clean": predict(detectors, origin01),
        "pgd": predict(detectors, denormalize(adv_norm).clamp(0, 1)),
        "plain_vae": predict(detectors, plain01),
        "final": predict(detectors, final01),
        "psnr": float(calculate_psnr(final_u8, origin_u8)),
        "ssim": float(calculate_ssim(final_u8, origin_u8)),
        "linf": float(np.abs(final_u8.astype(np.float32) - origin_u8.astype(np.float32)).max() / 255.0),
        "history": history,
    }
    return record


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--images_root", required=True, help="directory of AI-generated input images")
    parser.add_argument("--save_dir", required=True, help="output directory (created; must not exist unless --resume)")
    parser.add_argument("--pretrained_diffusion_path", default="stabilityai/stable-diffusion-2-1-base")
    parser.add_argument("--encoder_weights", default="./checkpoints/Controlvae.pt")
    parser.add_argument("--surrogates", default="E", help="comma list of E,R,D,S used in the attack loss")
    parser.add_argument("--evaluate", default="E,R,D,S", help="comma list of detectors reported per image")
    parser.add_argument("--loss", default="ce", choices=["ce", "margin"])
    parser.add_argument("--kappa", type=float, default=3.0, help="margin target for --loss margin")
    parser.add_argument("--decode_path", default="plain", choices=["plain", "final"])
    parser.add_argument("--lpips_weight", type=float, default=0.0)
    parser.add_argument("--di_prob", type=float, default=0.0)
    parser.add_argument("--iterations", type=int, default=10, help="LAO iterations (official code default 10)")
    parser.add_argument("--diffusion_steps", type=int, default=20)
    parser.add_argument("--start_step", type=int, default=18)
    parser.add_argument("--guidance", type=float, default=0.0)
    parser.add_argument("--lr", type=float, default=1e-2)
    parser.add_argument("--attack_loss_weight", type=float, default=10.0)
    parser.add_argument("--pgd_eps", type=float, default=4 / 255)
    parser.add_argument("--res", type=int, default=224)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--max_images", type=int, default=0, help="0 = all images in images_root")
    parser.add_argument("--resume", action="store_true", help="skip images that already have an output")
    parser.add_argument("--device", default="auto", choices=["auto", "cuda", "mps", "cpu"],
                        help="auto = cuda if available, else mps (Apple Silicon), else cpu")
    args = parser.parse_args(argv)

    m2_device.enable(args.device)
    seed_everything(args.seed)
    surrogate_names = [s.strip() for s in args.surrogates.split(",") if s.strip()]
    evaluate_names = [s.strip() for s in args.evaluate.split(",") if s.strip()]
    for n in surrogate_names + evaluate_names:
        if n not in DETECTOR_NAMES:
            raise ValueError(f"unknown detector {n}; choose from E,R,D,S")

    if os.path.isdir(args.save_dir) and os.listdir(args.save_dir) and not args.resume:
        raise FileExistsError(f"{args.save_dir} exists and is not empty; use a new save_dir or --resume")
    os.makedirs(args.save_dir, exist_ok=True)

    image_paths = natsorted([p for p in glob.glob(os.path.join(args.images_root, "*"))
                             if p.lower().endswith((".png", ".jpg", ".jpeg"))], alg=ns.PATH)
    if not image_paths:
        raise FileNotFoundError(f"no PNG/JPG images in {args.images_root}")
    if args.max_images > 0:
        image_paths = image_paths[: args.max_images]

    config = dict(vars(args))
    config.update({"n_images": len(image_paths), "torch": torch.__version__,
                   "command": " ".join(sys.argv)})
    with open(os.path.join(args.save_dir, "run_config.json"), "w") as f:
        json.dump(config, f, indent=2)

    print(f"[m2] loading Stable Diffusion from {args.pretrained_diffusion_path}", flush=True)
    sd_path = m2_device.local_model(args.pretrained_diffusion_path, ignore_fp16=(m2_device.DEVICE != "cuda"))
    model = StableDiffusionPipeline.from_pretrained(sd_path).to(m2_device.DEVICE)
    model.scheduler = DDIMScheduler.from_config(model.scheduler.config)
    model.unet.requires_grad_(False)
    model.vae.requires_grad_(False)
    model.text_encoder.requires_grad_(False)

    print(f"[m2] loading detectors {sorted(set(surrogate_names + evaluate_names))}", flush=True)
    detectors = {}
    for name in sorted(set(surrogate_names + evaluate_names)):
        det = other_attacks.model_selection(name).eval()
        det.requires_grad_(False)
        detectors[name] = det
    surrogate_models = [detectors[n] for n in surrogate_names]
    eval_detectors = {n: detectors[n] for n in evaluate_names}

    cvae = load_control_vae(model, args.encoder_weights)
    lpips_fn = build_lpips() if args.lpips_weight > 0 else None

    records_path = os.path.join(args.save_dir, "records.jsonl")
    done = set()
    if args.resume and os.path.exists(records_path):
        with open(records_path) as f:
            done = {json.loads(line)["image_id"] for line in f if line.strip()}

    for ind, path in enumerate(image_paths):
        image_id = str(ind).rjust(4, "0")
        out_prefix = os.path.join(args.save_dir, image_id)
        if image_id in done and os.path.exists(out_prefix + "_adv_image.png"):
            continue
        pil_image = Image.open(path).convert("RGB")
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.time()
        record = attack_one(model, pil_image, args, surrogate_models, eval_detectors, cvae, lpips_fn, out_prefix)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        record.update({"image_id": image_id, "source": os.path.basename(path),
                       "seconds": round(time.time() - t0, 2)})
        with open(records_path, "a") as f:
            f.write(json.dumps(record) + "\n")
        finals = " ".join(f"{k}:{'real' if v['pred'] == 1 else 'AI'}" for k, v in record["final"].items())
        print(f"[m2] {image_id} {record['seconds']:.1f}s psnr={record['psnr']:.2f} "
              f"ssim={record['ssim']:.3f} final-> {finals}", flush=True)

    print(f"[m2] done. outputs in {args.save_dir}", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/StealthDiffusion/m2_evaluate.py
"""
FIT5230 · Dark.ModelBreakers · Milestone 2 — evaluation of saved attack outputs.

Run from inside the official StealthDiffusion checkout with its Python 3.10 environment.

  python m2_evaluate.py --runs B00=results/B00 M2_full=results/M2_full \
        --real_dir data/real/test --out results/eval \
        --detectors E,R,D,S --conditions none,jpeg75,resize05

Everything is measured on the PNG files written by m2_stealth.py (re-loaded from disk),
never on in-memory tensors.  For every (run, detector, condition) it reports:

  n                 images in the run
  n_clean_detected  images the detector correctly flags as AI *before* the attack
                    (under the same post-processing condition)
  cond_asr          conditional attack success rate = among n_clean_detected, the fraction
                    whose attacked version is classified as real
  uncond_escape     attacked images classified as real / n  (no conditioning)
  real_fpr          real test photos classified as AI / n_real   (detector false positives)
  auc_real_vs_adv   ROC-AUC of the detector score separating real photos from attacked
                    images (1.0 = still perfectly separable by moving the threshold;
                    0.5 = attacked images are indistinguishable from real photos)
  auc_real_vs_clean the same for un-attacked generated images
  psnr/ssim/lpips/linf  distortion of the attacked image w.r.t. the 224x224 input
  plain_ok_final_fail  images where the ordinary-VAE decode evaded the detector but the
                    final Control-VAE output did not (motivation for --decode_path final)

Post-processing conditions (applied identically to clean and attacked images):
  none, jpeg<Q> (PIL JPEG quality Q), resize<S> (downscale by S/10 then back to 224,
  e.g. resize05 = 0.5x), combined with '+', e.g. jpeg75+resize05.
"""
import argparse
import csv
import io
import json
import os

import numpy as np
import torch
from PIL import Image
from natsort import natsorted, ns
from torchvision import transforms

import other_attacks  # noqa: E402
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import m2_device  # noqa: E402
from diff_latent_attack import calculate_psnr, calculate_ssim  # noqa: E402

normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
RES = 224


def apply_condition(pil, cond):
    for token in cond.split("+"):
        token = token.strip()
        if token == "none":
            continue
        if token.startswith("jpeg"):
            q = int(token[4:])
            buf = io.BytesIO()
            pil.save(buf, format="JPEG", quality=q)
            buf.seek(0)
            pil = Image.open(buf).convert("RGB")
        elif token.startswith("resize"):
            scale = int(token[6:]) / 10.0
            small = max(8, int(round(RES * scale)))
            pil = pil.resize((small, small), resample=Image.LANCZOS).resize((RES, RES), resample=Image.BICUBIC)
        else:
            raise ValueError(f"unknown condition token {token}")
    return pil


def pil_to_tensor(pil):
    return torch.from_numpy(np.float32(pil.convert("RGB")) / 255.0).permute(2, 0, 1)


@torch.no_grad()
def predict_batch(det, pils, batch_size=16):
    preds, margins = [], []
    for i in range(0, len(pils), batch_size):
        x = torch.stack([pil_to_tensor(p) for p in pils[i:i + batch_size]]).cuda()
        logits = det(normalize(x))
        preds += torch.argmax(logits, 1).tolist()
        margins += (logits[:, 1] - logits[:, 0]).tolist()
    return preds, margins


def auc(pos_scores, neg_scores):
    """ROC-AUC with real photos as positives (higher margin = more 'real')."""
    if len(pos_scores) == 0 or len(neg_scores) == 0:
        return float("nan")
    from sklearn.metrics import roc_auc_score
    y = [1] * len(pos_scores) + [0] * len(neg_scores)
    return float(roc_auc_score(y, list(pos_scores) + list(neg_scores)))


def load_run(run_dir):
    ids = natsorted([f[:-len("_originImage.png")] for f in os.listdir(run_dir) if f.endswith("_originImage.png")], alg=ns.PATH)
    items = []
    for i in ids:
        adv = os.path.join(run_dir, i + "_adv_image.png")
        if not os.path.exists(adv):
            continue
        plain = os.path.join(run_dir, i + "_plain_vae.png")
        items.append({"image_id": i,
                      "clean": Image.open(os.path.join(run_dir, i + "_originImage.png")).convert("RGB"),
                      "adv": Image.open(adv).convert("RGB"),
                      "plain": Image.open(plain).convert("RGB") if os.path.exists(plain) else None})
    seconds = {}
    rec = os.path.join(run_dir, "records.jsonl")
    if os.path.exists(rec):
        with open(rec) as f:
            for line in f:
                if line.strip():
                    r = json.loads(line)
                    seconds[r["image_id"]] = r.get("seconds")
    return items, seconds


def mean(xs):
    xs = [x for x in xs if x is not None and not (isinstance(x, float) and np.isnan(x))]
    return float(np.mean(xs)) if xs else float("nan")


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--runs", nargs="+", required=True, help="name=dir pairs")
    p.add_argument("--real_dir", required=True, help="directory of real 224x224 PNGs (test split)")
    p.add_argument("--out", required=True)
    p.add_argument("--detectors", default="E,R,D,S")
    p.add_argument("--conditions", default="none,jpeg75,resize05")
    p.add_argument("--no_lpips", action="store_true")
    p.add_argument("--device", default="auto", choices=["auto", "cuda", "mps", "cpu"],
                   help="auto = cuda if available, else mps (Apple Silicon), else cpu")
    args = p.parse_args(argv)

    m2_device.enable(args.device)
    os.makedirs(args.out, exist_ok=True)
    runs = {}
    for spec in args.runs:
        name, d = spec.split("=", 1)
        items, seconds = load_run(d)
        if not items:
            raise FileNotFoundError(f"no completed images in {d}")
        runs[name] = {"dir": d, "items": items, "seconds": seconds}
        print(f"[eval] run {name}: {len(items)} images from {d}", flush=True)
    real_files = natsorted([f for f in os.listdir(args.real_dir) if f.endswith(".png")], alg=ns.PATH)
    real_pils = [Image.open(os.path.join(args.real_dir, f)).convert("RGB") for f in real_files]
    print(f"[eval] {len(real_pils)} real test images", flush=True)
    conditions = [c.strip() for c in args.conditions.split(",") if c.strip()]

    # ---- distortion (condition-free), once per run ----
    lpips_fn = None
    if not args.no_lpips:
        from lossers.lpips import LPIPS
        lpips_fn = LPIPS(net="vgg", cache_dir="./.cache").cuda().eval()
    quality = {}
    for name, run in runs.items():
        for it in run["items"]:
            a, c = np.array(it["adv"]), np.array(it["clean"])
            q = {"psnr": float(calculate_psnr(a, c)), "ssim": float(calculate_ssim(a, c)),
                 "linf": float(np.abs(a.astype(np.float32) - c.astype(np.float32)).max() / 255.0), "lpips": None}
            if lpips_fn is not None:
                with torch.no_grad():
                    ta = (pil_to_tensor(it["adv"])[None].cuda() * 2 - 1)
                    tc = (pil_to_tensor(it["clean"])[None].cuda() * 2 - 1)
                    q["lpips"] = float(lpips_fn(tc, ta).mean().item())
            quality[(name, it["image_id"])] = q
    del lpips_fn
    torch.cuda.empty_cache()

    per_image, real_rows, summary = [], [], []
    for det_name in [d.strip() for d in args.detectors.split(",")]:
        det = other_attacks.model_selection(det_name).eval()
        det.requires_grad_(False)
        for cond in conditions:
            r_pred, r_margin = predict_batch(det, [apply_condition(p, cond) for p in real_pils])
            for f, pr, mg in zip(real_files, r_pred, r_margin):
                real_rows.append({"detector": det_name, "condition": cond, "image": f, "pred": pr, "real_margin": mg})
            real_fpr = float(np.mean([pr == 0 for pr in r_pred])) if r_pred else float("nan")
            for name, run in runs.items():
                items = run["items"]
                c_pred, c_margin = predict_batch(det, [apply_condition(it["clean"], cond) for it in items])
                a_pred, a_margin = predict_batch(det, [apply_condition(it["adv"], cond) for it in items])
                has_plain = all(it["plain"] is not None for it in items)
                if has_plain:
                    p_pred, p_margin = predict_batch(det, [apply_condition(it["plain"], cond) for it in items])
                else:
                    p_pred, p_margin = [None] * len(items), [None] * len(items)
                n = len(items)
                detected = [i for i in range(n) if c_pred[i] == 0]
                evaded = [i for i in detected if a_pred[i] == 1]
                plain_ok_final_fail = sum(1 for i in detected if has_plain and p_pred[i] == 1 and a_pred[i] == 0)
                for i, it in enumerate(items):
                    q = quality[(name, it["image_id"])] if cond == "none" else {"psnr": None, "ssim": None, "linf": None, "lpips": None}
                    per_image.append({"run": name, "detector": det_name, "condition": cond, "image_id": it["image_id"],
                                      "clean_pred": c_pred[i], "clean_margin": c_margin[i],
                                      "adv_pred": a_pred[i], "adv_margin": a_margin[i],
                                      "plain_pred": p_pred[i], "plain_margin": p_margin[i],
                                      "psnr": q["psnr"], "ssim": q["ssim"], "linf": q["linf"], "lpips": q["lpips"],
                                      "seconds": run["seconds"].get(it["image_id"])})
                row = {"run": name, "detector": det_name, "condition": cond, "n": n,
                       "n_clean_detected": len(detected),
                       "cond_asr": (len(evaded) / len(detected)) if detected else float("nan"),
                       "uncond_escape": float(np.mean([pr == 1 for pr in a_pred])),
                       "mean_adv_margin": float(np.mean(a_margin)),
                       "mean_clean_margin": float(np.mean(c_margin)),
                       "real_fpr": real_fpr,
                       "auc_real_vs_adv": auc(r_margin, a_margin),
                       "auc_real_vs_clean": auc(r_margin, c_margin),
                       "plain_ok_final_fail": plain_ok_final_fail if has_plain else None,
                       "psnr": mean([quality[(name, it["image_id"])]["psnr"] for it in items]),
                       "ssim": mean([quality[(name, it["image_id"])]["ssim"] for it in items]),
                       "lpips": mean([quality[(name, it["image_id"])]["lpips"] for it in items]),
                       "linf": mean([quality[(name, it["image_id"])]["linf"] for it in items]),
                       "seconds": mean([run["seconds"].get(it["image_id"]) for it in items])}
                summary.append(row)
                print(f"[eval] {name:<12} {det_name} {cond:<16} detected {len(detected):>3}/{n:<3} "
                      f"cond_ASR {row['cond_asr']:.2f}  escape {row['uncond_escape']:.2f}  "
                      f"real_FPR {real_fpr:.2f}  AUC {row['auc_real_vs_adv']:.2f}", flush=True)
        del det
        torch.cuda.empty_cache()

    def write_csv(path, rows):
        if not rows:
            return
        with open(path, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader()
            w.writerows(rows)

    write_csv(os.path.join(args.out, "per_image.csv"), per_image)
    write_csv(os.path.join(args.out, "real_predictions.csv"), real_rows)
    write_csv(os.path.join(args.out, "summary.csv"), summary)
    with open(os.path.join(args.out, "summary.json"), "w") as f:
        json.dump({"runs": {k: v["dir"] for k, v in runs.items()}, "real_dir": args.real_dir,
                   "conditions": conditions, "summary": summary}, f, indent=2)
    print(f"[eval] wrote {args.out}/summary.csv, per_image.csv, real_predictions.csv", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/StealthDiffusion/m2_challenge.py
"""
FIT5230 · Dark.ModelBreakers · Milestone 2 — challenge package for other teams.

  python m2_challenge.py build --real_dir data/real/test --gen_dir data/gen/test \
        --attack_dir results/M2_full --n_per_group 30 --out challenge --seed 5230

Builds
  challenge/public/images/<random id>.png    shuffled mix of real photos, raw generated
                                             images and our attacked images (labels hidden)
  challenge/public/README.md                 task + submission format
  challenge/public/submission_template.csv
  challenge/challenge_public.zip             <- share only this
  challenge/private/challenge_key.csv        <- never share

  python m2_challenge.py score --key challenge/private/challenge_key.csv --submission their.csv
Scores a returned CSV (image_id, pred_label with 1 = generated, 0 = real): overall accuracy,
detection rate on raw generated images, detection rate on attacked images, false-positive
rate on real photos.
"""
import argparse
import csv
import hashlib
import json
import os
import random
import shutil
import zipfile

from natsort import natsorted, ns

README = """# Dark.ModelBreakers — Milestone 2 challenge: can you detect our stealthed images?

`images/` contains {n_total} PNG images (224x224) in random order:
- real photographs,
- AI-generated images (Stable Diffusion, unmodified),
- AI-generated images processed by our modified StealthDiffusion attack.

Task: label every image as **1 = generated (AI)** or **0 = real**.
Fill `submission_template.csv` (columns `image_id,pred_label`) and send it back to us on Ed;
we run the scorer against the private key and post: overall accuracy, detection rate on raw
generated images, detection rate on attacked images, and false-positive rate on real photos.

If you want a white-box / adaptive comparison instead, share a detector wrapper with us:
`load_detector() -> torch.nn.Module` whose forward takes float RGB `[B,3,H,W]` in `[0,1]`
and returns fake logits `[B]` (positive = generated).
"""


def cmd_build(args):
    random.seed(args.seed)
    groups = {
        "real": [os.path.join(args.real_dir, f) for f in natsorted(os.listdir(args.real_dir), alg=ns.PATH) if f.endswith(".png")],
        "generated": [os.path.join(args.gen_dir, f) for f in natsorted(os.listdir(args.gen_dir), alg=ns.PATH) if f.endswith(".png")],
        "attacked": [os.path.join(args.attack_dir, f) for f in natsorted(os.listdir(args.attack_dir), alg=ns.PATH) if f.endswith("_adv_image.png")],
    }
    public_img = os.path.join(args.out, "public", "images")
    private = os.path.join(args.out, "private")
    if os.path.exists(os.path.join(args.out, "public")):
        raise FileExistsError(f"{args.out}/public exists; choose a new --out")
    os.makedirs(public_img)
    os.makedirs(private, exist_ok=True)
    key_rows = []
    for group, files in groups.items():
        if len(files) < args.n_per_group:
            raise ValueError(f"group {group} has only {len(files)} images (< {args.n_per_group})")
        chosen = random.sample(files, args.n_per_group)
        for src in chosen:
            image_id = "img_" + hashlib.sha256((str(args.seed) + src).encode()).hexdigest()[:10]
            shutil.copyfile(src, os.path.join(public_img, image_id + ".png"))
            key_rows.append({"image_id": image_id, "group": group, "label": 0 if group == "real" else 1, "source": src})
    random.shuffle(key_rows)
    with open(os.path.join(private, "challenge_key.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["image_id", "group", "label", "source"])
        w.writeheader()
        w.writerows(key_rows)
    with open(os.path.join(args.out, "public", "submission_template.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["image_id", "pred_label"])
        for r in natsorted(key_rows, key=lambda r: r["image_id"], alg=ns.PATH):
            w.writerow([r["image_id"], ""])
    with open(os.path.join(args.out, "public", "README.md"), "w") as f:
        f.write(README.format(n_total=len(key_rows)))
    zip_path = os.path.join(args.out, "challenge_public.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(os.path.join(args.out, "public")):
            for fn in files:
                full = os.path.join(root, fn)
                z.write(full, os.path.relpath(full, os.path.join(args.out, "public")))
    print(f"[challenge] {len(key_rows)} images -> {zip_path} (share this); key in {private}/challenge_key.csv (private)")


def cmd_score(args):
    with open(args.key) as f:
        key = {r["image_id"]: r for r in csv.DictReader(f)}
    with open(args.submission) as f:
        sub = {r["image_id"]: r for r in csv.DictReader(f)}
    missing = [i for i in key if i not in sub or sub[i].get("pred_label", "") == ""]
    stats = {g: {"n": 0, "correct": 0} for g in ("real", "generated", "attacked")}
    for image_id, r in key.items():
        if image_id in missing:
            continue
        pred = int(float(sub[image_id]["pred_label"]))
        g = stats[r["group"]]
        g["n"] += 1
        g["correct"] += int(pred == int(r["label"]))
    n = sum(g["n"] for g in stats.values())
    result = {
        "team": args.team,
        "n_scored": n, "n_missing": len(missing),
        "overall_accuracy": sum(g["correct"] for g in stats.values()) / n if n else None,
        "detection_rate_generated": stats["generated"]["correct"] / stats["generated"]["n"] if stats["generated"]["n"] else None,
        "detection_rate_attacked": stats["attacked"]["correct"] / stats["attacked"]["n"] if stats["attacked"]["n"] else None,
        "false_positive_rate_real": 1 - stats["real"]["correct"] / stats["real"]["n"] if stats["real"]["n"] else None,
    }
    print(json.dumps(result, indent=2))
    if args.save:
        with open(args.save, "w") as f:
            json.dump(result, f, indent=2)


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="mode", required=True)
    b = sub.add_parser("build")
    b.add_argument("--real_dir", required=True)
    b.add_argument("--gen_dir", required=True)
    b.add_argument("--attack_dir", required=True)
    b.add_argument("--n_per_group", type=int, default=30)
    b.add_argument("--out", default="challenge")
    b.add_argument("--seed", type=int, default=5230)
    s = sub.add_parser("score")
    s.add_argument("--key", required=True)
    s.add_argument("--submission", required=True)
    s.add_argument("--team", default="unknown")
    s.add_argument("--save", default="")
    args = p.parse_args(argv)
    {"build": cmd_build, "score": cmd_score}[args.mode](args)


if __name__ == "__main__":
    main()

In [ ]:
#@title 3.6  Syntax check of the four scripts
for name in ['m2_device.py', 'm2_data.py', 'm2_stealth.py', 'm2_evaluate.py', 'm2_challenge.py']:
    subprocess.run([PYTHON, '-m', 'py_compile', name], check=True)
    print('ok', name)

## 4. Data
The official evaluation uses GenImage (100 GB per generator). For M2 we build a small, fully documented set:

- **Real photos:** Imagenette (public 10-class ImageNet subset), validation images, centre-cropped to 224x224.
- **AI images:** Stable Diffusion, prompt `"a photo of a <class>"` for the same 10 classes, 512x512, then resized to 224x224 (the same resolution the official attack uses). GenImage's SD subsets were built the same way, so the official detectors should recognise them.
- `dev` split (10 + 10 images) is only used to check the detectors; `test` split is used for all results.

Every image has a manifest line (source file, or prompt + seed + model).

In [ ]:
#@title 4.1  Build the dataset (real photos + generated images)
N_DEV = 10      #@param {type:"integer"}
N_TEST = 100    #@param {type:"integer"}
GEN_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"   #@param {type:"string"}
SD_MODEL  = "Manojb/stable-diffusion-2-1-base"              #@param {type:"string"}
REAL_SRC  = ""  #@param {type:"string"}
# REAL_SRC: leave empty to download Imagenette automatically. If the download fails
# (or you prefer your own real photos), set it to a folder of real images on Drive,
# e.g. a GenImage ".../val/nature" folder; it is searched recursively.
DATA = Path('/content/StealthDiffusion/data')

def run_step(cmd):
    # Show the child's real output/error inline instead of a bare CalledProcessError.
    r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(r.stdout)
    return r.returncode

if not (DATA / 'real' / 'test' / 'manifest.csv').exists():
    cmd = [PYTHON, 'm2_data.py', 'real', '--out', str(DATA), '--n_dev', str(N_DEV), '--n_test', str(N_TEST)]
    if REAL_SRC.strip():
        cmd += ['--real_src', REAL_SRC.strip()]
    if run_step(cmd) != 0:
        raise RuntimeError('real-image step failed; read the output above. '
                           'If it is a download error, set REAL_SRC to a folder of real images on Drive and re-run.')
if not (DATA / 'gen' / 'test' / 'manifest.csv').exists():
    if run_step([PYTHON, 'm2_data.py', 'generate', '--out', str(DATA), '--n_dev', str(N_DEV), '--n_test', str(N_TEST),
                 '--model', GEN_MODEL]) != 0:
        print(f'\n!!! generation with {GEN_MODEL} failed (see output above); retrying with {SD_MODEL}. '
              'The manifest records which model produced the images.')
        if run_step([PYTHON, 'm2_data.py', 'generate', '--out', str(DATA), '--n_dev', str(N_DEV), '--n_test', str(N_TEST),
                     '--model', SD_MODEL]) != 0:
            raise RuntimeError('image generation failed with both models; read the output above.')
import pandas as pd
print(pd.read_csv(DATA / 'gen' / 'test' / 'manifest.csv').head(3).to_string())
print('real test:', len(list((DATA / 'real' / 'test').glob('*.png'))), '| generated test:', len(list((DATA / 'gen' / 'test').glob('*.png'))))

In [ ]:
#@title 4.2  Detector sanity check on the dev split (official label convention: 0 = AI, 1 = real)
subprocess.run([PYTHON, 'm2_data.py', 'check', '--out', str(DATA)], check=True)
dev_check = pd.read_csv(DATA / 'dev_check.csv')
display(dev_check)
assert (dev_check.acc_gen >= 0.5).all() and (dev_check.acc_real >= 0.5).all(), 'a detector does not work on this data; check checkpoints / polarity before attacking'

In [ ]:
#@title 4.3  A few samples
import matplotlib.pyplot as plt
from PIL import Image
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for j in range(5):
    axes[0, j].imshow(Image.open(DATA / 'real' / 'test' / f'{j:04d}.png')); axes[0, j].set_title(f'real {j:04d}', fontsize=9)
    axes[1, j].imshow(Image.open(DATA / 'gen' / 'test' / f'{j:04d}.png'));  axes[1, j].set_title(f'generated {j:04d}', fontsize=9)
for a in axes.ravel(): a.axis('off')
plt.suptitle('Figure 1. Test images: real photos (top) and Stable Diffusion images (bottom)'); plt.tight_layout(); plt.show()

## 5. Attack runs
Each variant attacks the same `N_ATTACK` generated test images with the same seed. Outputs go to Drive (`DRIVE_ROOT/runs/<variant>`) so that an interrupted session can be resumed (`--resume` skips finished images).

Per image the script saves the 224x224 input (`_originImage.png`), the ordinary-VAE decode of the final latent (`_plain_vae.png`, diagnostic) and the final Control-VAE output (`_adv_image.png`, the attack result), plus `records.jsonl` with the loss history and the time per image.

In [ ]:
#@title 5.1  Run the four variants
N_ATTACK = 20   #@param {type:"integer"}
RUNS = Path(DRIVE_ROOT) / 'runs'
VARIANTS = {
    'B00':  [],                                                     # official pipeline
    'B10':  ['--decode_path', 'final'],                              # loss on the final Control-VAE path
    'ENS':  ['--surrogates', 'E,R,D'],                               # surrogate ensemble
    'FULL': ['--surrogates', 'E,R,D', '--decode_path', 'final', '--loss', 'margin',
             '--lpips_weight', '1.0', '--di_prob', '0.5'],           # all M2 changes
}
common = ['--images_root', str(DATA / 'gen' / 'test'), '--max_images', str(N_ATTACK),
          '--pretrained_diffusion_path', SD_MODEL, '--encoder_weights', './checkpoints/Controlvae.pt',
          '--iterations', '10', '--seed', '42', '--resume']
import time
for name, extra in VARIANTS.items():
    t0 = time.time()
    print(f'=== variant {name}: m2_stealth.py {" ".join(extra)} ===', flush=True)
    subprocess.run([PYTHON, '-u', 'm2_stealth.py', '--save_dir', str(RUNS / name), *common, *extra], check=True)
    print(f'=== {name} finished in {(time.time() - t0) / 60:.1f} min ===\n', flush=True)

In [ ]:
#@title 5.2  Quick look at the run logs (in-memory predictions; the real evaluation is Section 6)
import json
rows, secs = [], {}
for name in VARIANTS:
    with open(RUNS / name / 'records.jsonl') as f:
        recs = [json.loads(l) for l in f if l.strip()]
    secs[name] = sum(r['seconds'] for r in recs) / len(recs)
    for det in ['E', 'R', 'D', 'S']:
        rows.append({'variant': name, 'detector': det,
                     'clean->AI': sum(r['clean'][det]['pred'] == 0 for r in recs) / len(recs),
                     'final->real': sum(r['final'][det]['pred'] == 1 for r in recs) / len(recs)})
display(pd.DataFrame(rows).pivot_table(index='variant', columns='detector', values=['clean->AI', 'final->real']).round(2))
print('seconds per image:', {k: round(v, 1) for k, v in secs.items()})

## 6. Evaluation
All numbers below are computed on the **saved PNG files** by `m2_evaluate.py`, for four detectors and three post-processing conditions (`none`, `jpeg75`, `resize05` = downscale to 112 px and back).

Definitions (per variant, detector, condition):
- **cond. ASR** — among generated images the detector correctly flags as AI *before* the attack (under the same condition), the fraction whose attacked version is classified as real. This is the paper's ASR with an explicit denominator.
- **real FPR** — real test photos classified as AI. Shows how strict each detector is; the attack is only meaningful if this is low.
- **AUC (real vs attacked)** — can the detector still separate real photos from attacked images by moving its threshold? 0.5 = no.
- **PSNR / SSIM / LPIPS / Linf** — distortion between the attacked image and the 224x224 input.
- Detectors E, R, D are white-box for the ENS/FULL variants; **S is held out for every variant** (transfer). For B00/B10 only E is white-box, so R, D, S are transfer targets.

In [ ]:
#@title 6.1  Evaluate all runs
EVAL = Path(DRIVE_ROOT) / 'eval'
subprocess.run([PYTHON, '-u', 'm2_evaluate.py', '--runs', *[f'{n}={RUNS / n}' for n in VARIANTS],
                '--real_dir', str(DATA / 'real' / 'test'), '--out', str(EVAL),
                '--detectors', 'E,R,D,S', '--conditions', 'none,jpeg75,resize05'], check=True)
summary = pd.read_csv(EVAL / 'summary.csv')
per_image = pd.read_csv(EVAL / 'per_image.csv')
summary.head()

In [ ]:
#@title 6.2  Table 1 — conditional ASR without post-processing (white-box cells marked *)
WHITE_BOX = {'B00': ['E'], 'B10': ['E'], 'ENS': ['E', 'R', 'D'], 'FULL': ['E', 'R', 'D']}
t = summary[summary.condition == 'none'].pivot(index='run', columns='detector', values='cond_asr').reindex(index=list(VARIANTS), columns=['E', 'R', 'D', 'S'])
fmt = t.copy().astype(object)
for run in t.index:
    for det in t.columns:
        v = t.loc[run, det]
        fmt.loc[run, det] = ('%.2f' % v) + ('*' if det in WHITE_BOX[run] else '')
fmt['transfer mean'] = [('%.2f' % t.loc[run, [d for d in t.columns if d not in WHITE_BOX[run]]].mean()) for run in t.index]
print('Table 1. Conditional ASR (higher = attack better). * = detector was in the attack loss (white-box).')
display(fmt)
den = summary[summary.condition == 'none'].pivot(index='run', columns='detector', values='n_clean_detected').reindex(index=list(VARIANTS), columns=['E', 'R', 'D', 'S'])
print('denominators (images detected as AI before the attack):'); display(den)
print('real-photo false positive rate per detector:'); display(summary[(summary.condition == 'none') & (summary.run == 'B00')].set_index('detector')['real_fpr'].round(3).to_frame().T)

In [ ]:
#@title 6.3  Table 2 — robustness to JPEG and resizing (transfer detector S and mean over R, D, S)
rob = summary.pivot(index=['run', 'condition'], columns='detector', values='cond_asr').reindex(list(VARIANTS), level=0).reindex(columns=['E', 'R', 'D', 'S']).round(2)
print('Table 2. Conditional ASR under post-processing (both clean and attacked images are processed the same way).')
display(rob)
auc = summary.pivot(index=['run', 'condition'], columns='detector', values='auc_real_vs_adv').reindex(list(VARIANTS), level=0).reindex(columns=['E', 'R', 'D', 'S']).round(2)
print('Table 3. AUC real-vs-attacked (0.5 = the detector can no longer separate them by moving its threshold).')
display(auc)

In [ ]:
#@title 6.4  Table 4 — image quality, cost and plain-vs-final regressions
q = summary[(summary.condition == 'none') & (summary.detector == 'E')].set_index('run')[['psnr', 'ssim', 'lpips', 'linf', 'seconds']].reindex(list(VARIANTS)).round(3)
q.columns = ['PSNR (dB)', 'SSIM', 'LPIPS', 'Linf', 'sec / image']
print('Table 4. Distortion of the saved attack image w.r.t. the 224x224 input, and time per image on this GPU.')
display(q)
reg = summary[summary.condition == 'none'].pivot(index='run', columns='detector', values='plain_ok_final_fail').reindex(index=list(VARIANTS), columns=['E', 'R', 'D', 'S'])
print('Table 5. Images where the ordinary-VAE decode evaded the detector but the final Control-VAE output did not (motivation for --decode_path final).')
display(reg)

In [ ]:
#@title 6.5  Figure 2 — ASR per detector and variant; Figure 3 — robustness
import numpy as np
dets = ['E', 'R', 'D', 'S']
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
w = 0.2
for i, run in enumerate(VARIANTS):
    vals = [summary[(summary.run == run) & (summary.condition == 'none') & (summary.detector == d)].cond_asr.iloc[0] for d in dets]
    ax[0].bar(np.arange(4) + (i - 1.5) * w, vals, w, label=run)
ax[0].set_xticks(np.arange(4)); ax[0].set_xticklabels(['E (EffNet-B0)', 'R (ResNet-50)', 'D (DeiT-B)', 'S (Swin, held out)'])
ax[0].set_ylabel('conditional ASR'); ax[0].set_ylim(0, 1.05); ax[0].legend(fontsize=8); ax[0].set_title('Figure 2. ASR per detector (no post-processing)')
conds = ['none', 'jpeg75', 'resize05']
for i, run in enumerate(VARIANTS):
    vals = [summary[(summary.run == run) & (summary.condition == c) & (summary.detector == 'S')].cond_asr.iloc[0] for c in conds]
    ax[1].bar(np.arange(3) + (i - 1.5) * w, vals, w, label=run)
ax[1].set_xticks(np.arange(3)); ax[1].set_xticklabels(conds); ax[1].set_ylim(0, 1.05); ax[1].legend(fontsize=8)
ax[1].set_title('Figure 3. Transfer ASR on S under post-processing')
plt.tight_layout(); plt.show()

In [ ]:
#@title 6.6  Figure 4 — examples (input / ordinary-VAE decode / final output) with detector verdicts
from PIL import Image
def verdicts(run, image_id, col):
    v = per_image[(per_image.run == run) & (per_image.condition == 'none') & (per_image.image_id.astype(str).str.zfill(4) == image_id)]
    return ' '.join(f"{r.detector}:{'real' if getattr(r, col) == 1 else 'AI'}" for r in v.itertuples())
show_runs = ['B00', 'FULL']
ids = sorted(p.name[:4] for p in (RUNS / 'B00').glob('*_originImage.png'))[:3]
fig, axes = plt.subplots(len(ids) * len(show_runs), 3, figsize=(9, 3.2 * len(ids) * len(show_runs)), squeeze=False)
r = 0
for image_id in ids:
    for run in show_runs:
        for j, (suffix, col, title) in enumerate([('_originImage.png', 'clean_pred', 'input'), ('_plain_vae.png', 'plain_pred', 'ordinary VAE decode'), ('_adv_image.png', 'adv_pred', 'final output')]):
            axes[r, j].imshow(Image.open(RUNS / run / f'{image_id}{suffix}'))
            axes[r, j].set_title(f'{run} {image_id} {title}\n{verdicts(run, image_id, col)}', fontsize=8); axes[r, j].axis('off')
        r += 1
plt.suptitle('Figure 4. Examples. Verdicts of the four detectors on the saved PNGs.'); plt.tight_layout(); plt.show()

In [ ]:
#@title 6.7  Figure 5 — optimisation curves (surrogate margin per LAO iteration)
import numpy as np
fig, ax = plt.subplots(1, len(VARIANTS), figsize=(4 * len(VARIANTS), 3.2), sharey=True)
for a, name in zip(np.atleast_1d(ax), VARIANTS):
    with open(RUNS / name / 'records.jsonl') as f:
        recs = [json.loads(l) for l in f if l.strip()]
    for r in recs[:10]:
        a.plot([np.mean(h['margins']) for h in r['history']], alpha=0.6)
    a.axhline(0, color='k', lw=0.8); a.set_title(f'{name}: mean surrogate margin (real - AI)', fontsize=9); a.set_xlabel('LAO iteration')
plt.suptitle('Figure 5. Above 0 = the surrogate(s) call the decoded image real'); plt.tight_layout(); plt.show()

## 7. Challenge package for other teams
> **Can you detect our stealthed images?**

The package mixes real photos, unmodified Stable Diffusion images and images processed by our FULL attack (labels hidden). A team returns a CSV with one label per image; we run the scorer and post detection rate on attacked images, detection rate on raw generated images and false-positive rate on real photos. Teams that prefer a white-box comparison can send us a detector wrapper (`load_detector()` returning a `torch.nn.Module` that maps RGB `[B,3,H,W]` in `[0,1]` to fake logits `[B]`); our evaluation script then treats it as one more detector.

In [ ]:
#@title 7.1  Build the package (public zip goes to Ed; the key stays private on Drive)
CHALLENGE = Path(DRIVE_ROOT) / 'challenge'
if not (CHALLENGE / 'challenge_public.zip').exists():
    n_per_group = min(30, N_ATTACK)
    subprocess.run([PYTHON, 'm2_challenge.py', 'build', '--real_dir', str(DATA / 'real' / 'test'), '--gen_dir', str(DATA / 'gen' / 'test'),
                    '--attack_dir', str(RUNS / 'FULL'), '--n_per_group', str(n_per_group), '--out', str(CHALLENGE), '--seed', '5230'], check=True)
print(open(CHALLENGE / 'public' / 'README.md').read())
print('public zip:', CHALLENGE / 'challenge_public.zip')
print('scorer   : python m2_challenge.py score --key', CHALLENGE / 'private' / 'challenge_key.csv', '--submission <their.csv> --team <name>')

## 8. Engagement with another team
*(fill in before posting)*

- **Target team / post:** `[Light team name, Ed link]`
- **What we target:** `[e.g. their detector's reliance on frequency evidence / their purification step]`
- **What we sent:** the challenge zip (Section 7) and the detector interface above, on `[date]`.
- **What came back / our response:** `[their prediction CSV and score, or their detector run through m2_evaluate.py as an extra detector; what we changed after seeing it]`

## 9. Limitations and next steps
- Data: 10 classes, one generator, `N_ATTACK` images. GenImage-scale numbers are out of reach in the M2 window; the paper's numbers are context, not our baseline. Our baseline is B00 run on the same images.
- Detectors E, R, D are white-box for ENS/FULL. Only S (and JPEG/resize) measures transfer. Adding a public CLIP-based detector (UnivFD) as a held-out target is the next step.
- The Control-VAE is used as released; we did not retrain it. The initial PGD epsilon does not bound the final output, so we report the measured Linf/PSNR/SSIM/LPIPS of the saved image instead of claiming a budget.
- Input diversity and the margin loss add hyper-parameters (`di_prob`, `kappa`) that were set once and not tuned on the test split.
- Next (M3): EOT with a differentiable JPEG inside the LAO loop, a CLIP feature objective for semantic detectors, and query-based refinement against the Light team's detector.

## 10. References
1. Z. Zhou et al. *StealthDiffusion: Towards Evading Diffusion Forensic Detection through Diffusion Model.* ACM MM 2024. arXiv:2408.05669.
2. M. Zhu et al. *GenImage: A Million-Scale Benchmark for Detecting AI-Generated Images.* NeurIPS 2023.
3. A. Madry et al. *Towards Deep Learning Models Resistant to Adversarial Attacks.* ICLR 2018.
4. C. Xie et al. *Improving Transferability of Adversarial Examples with Input Diversity.* CVPR 2019.
5. N. Carlini, D. Wagner. *Towards Evaluating the Robustness of Neural Networks.* IEEE S&P 2017 (margin loss).
6. R. Zhang et al. *The Unreasonable Effectiveness of Deep Features as a Perceptual Metric.* CVPR 2018 (LPIPS).
7. Y. Liu et al. *Delving into Transferable Adversarial Examples and Black-box Attacks.* ICLR 2017 (ensemble attacks).
8. J. Howard. *Imagenette.* https://github.com/fastai/imagenette